# Hands-On: HR Policy Retrieval-Augmented Generation (RAG) System

### Name - Anik Chowdhury

This notebook demonstrates a complete RAG system pipeline. It retrieves relevant information from an HR Policy Handbook and generates accurate answers using Artificial Intelligence.

### Step 1: Import Required Libraries

We begin by importing the necessary modules. This includes tools for loading documents, splitting text, creating embeddings, and setting up the LLM (Large Language Model).

In [21]:
import os
from dotenv import load_dotenv

# LangChain Document Loaders & Splitters
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings & Vector Store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# LLM (Groq)
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

# Prompts, Parsers & Runnables for LCEL
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

### Step 2: Load Environment Variables

We load the API keys securely from a `.env` file to authenticate external AI services.

**API Keys Configuration:**
- **Required Keys:** `GROQ_API_KEY` and `GOOGLE_API_KEY` are mandatory because this notebook runs on Groq LLM and Google Generative AI embeddings by default.
- **Optional Key (`HUGGINGFACEHUB_API_TOKEN`):** The assertion check for the Hugging Face token is commented out by default. This avoids unnecessary errors if you do not have a Hugging Face account. Uncomment this check only if you choose to use Hugging Face embeddings.

In [ ]:
#from pathlib import Path
#from dotenv import find_dotenv


#env_file = find_dotenv(".env", usecwd=True)
#print("Loaded:", env_file)

#load_dotenv(env_file, override=True)

#assert os.getenv("GROQ_API_KEY") is not None, "GROQ_API_KEY Not Found"
#assert os.getenv("GOOGLE_API_KEY") is not None, "GOOGLE_API_KEY Not Found"
#assert os.getenv("HUGGINGFACEHUB_API_TOKEN") is not None, "HUGGINGFACEHUB_API_TOKEN Not Found"

### Step 2: Load Environment Variables

We load the required API keys from the local `.env` file into the environment. 

**Note:** Ensure your `.env` file contains valid keys for `GROQ_API_KEY` and `GOOGLE_API_KEY` to authenticate the models used in this notebook.

In [36]:
load_dotenv()
print("Environment variables loaded successfully.")

Environment variables loaded successfully.


### Step 3: Define the Document Path

We specify the file path to our HR Policy Handbook PDF document.

**Note:** If the system cannot locate the document or throws a file not found error, please verify that the file exists and provide the correct relative or absolute path to your PDF file.

In [23]:
pdf_path = "ABC_Pvt_Ltd_Policy_Handbook.pdf"
# fallback path check
if not os.path.exists(pdf_path):
    pdf_path = "../../ABC_Pvt_Ltd_Policy_Handbook.pdf"

### Step 4: Load the PDF Document

We use a document loader to read the PDF file and extract its text content.

In [24]:
loader = PyPDFLoader(pdf_path)
docs = loader.load()

incorrect startxref pointer(1)
parsing for Object Streams


### Step 5: Split Text into Chunks

Large documents are difficult for AI models to process all at once. We break the document down into smaller chunks of 1000 characters. We also keep a 200-character overlap to maintain the context between these chunks.

In [25]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)
print(f"Total document chunks created: {len(splits)}")

Total document chunks created: 81


### Step 6: Initialize the Embedding Model

Embeddings convert text into numerical formats so the AI can understand meaning and context.

**Current Setup:** We are currently using **Google Gemini Embeddings** (`gemini-embedding-2`) by default.

**How to Change Models:** If the current model fails or you want to use HuggingFace instead, you can switch it easily. Simply add a `#` sign before the `GoogleGenerativeAIEmbeddings` code lines to comment them out. Then, remove the `#` signs from one of the `HuggingFaceEmbeddings` blocks above it.

In [26]:
#embeddings = HuggingFaceEmbeddings(
#    model_name="sentence-transformers/all-MiniLM-L6-v2"
#)

#embeddings = HuggingFaceEndpointEmbeddings(
#    model="sentence-transformers/all-MiniLM-L6-v2"
#)


embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

### Step 7: Create the Vector Store

We save our processed text chunks into a **FAISS** vector database. This database helps the system search for relevant information very quickly.

In [27]:
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

### Step 8: Configure the Retriever

We set up the vector store to act as a retriever. It will fetch the top 3 most relevant sections from the document based on the user's question.

In [28]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### Step 9: Initialize the Large Language Model (LLM)

This is the main AI model that will read the retrieved text and generate a smart, readable answer.

**Current Setup:** We are currently using **ChatGroq** as our default LLM.

**How to Change Models:** If you face any issues with Groq, you can use the Google Gemini model as a backup. Just add a `#` sign before the `ChatGroq` lines to comment them out. After that, remove the `#` signs from the `ChatGoogleGenerativeAI` block.

In [29]:
llm = ChatGroq(
    temperature=0.5, 
    model_name="openai/gpt-oss-20b"
)


#llm = ChatGoogleGenerativeAI(
#    model = "gemini-3.6-flash"
#)

### Step 10: Define the Prompt Template

We give strict instructions to the AI using a system prompt. It tells the AI to use only the provided context and keep the final answer professional and concise.

In [30]:
system_prompt = (
    "You are an assistant for HR policy question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Keep the answer concise, clear and professional."
    "\n\n"
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

### Step 11: Set Up the Output Parser

We use an output parser to extract clean text from the raw response generated by the AI.

In [31]:
output_parser = StrOutputParser()

### Step 12: Define Document Formatting

We create a small helper function to join the retrieved text chunks together into a single readable paragraph.

In [32]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### Step 13: Build the RAG Chain

We connect all the previous steps together. The system will now sequentially retrieve text, format it, pass it to the prompt, and generate an answer through the LLM.

In [33]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

### Step 14: Ask the Question

We define our query and run the complete chain to find the answer from the HR policy.

In [34]:
# Query from the policy document
question = "What is Vision, Mission and Core Values"

response = chain.invoke(question)

### Step 15: Display the Final Result

We print the user's original question alongside the final answer generated by our AI system.

In [35]:
print("Question:", question)
print("\nAnswer:\n", response)

Question: What is Vision, Mission and Core Values

Answer:
 **Vision**  
“To be the most trusted engineering partner for organisations that want to turn data into decisions.”

**Mission**  
“To deliver reliable, secure and human‑centred technology while creating a workplace where every employee can learn, lead and grow.”

**Core Values (C.R.A.F.T.)**  
| Value | Essence |
|-------|---------|
| **Curiosity** | We ask why and keep learning. |
| **Responsibility** | We own outcomes for clients, colleagues and the community. |
| **Accountability & Integrity** | We do what we say, and we say what is true. |
| **Fairness & Respect** | We treat every person with dignity, irrespective of background. |
| **Teamwork** | We win together; credit is shared and failures are learned from, not blamed. |

These principles guide ABC Private Limited’s culture, decision‑making, and interactions both internally and with clients.
